In [1]:
!pip install pandas scikit-learn sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.4 MB/s eta 0:00:00


In [ ]:
from google.colab import ai
import pandas as pd
import os
import json
import re
import time

TRAIN_SIZE = 200
EVAL_SIZE = 50
BATCH_SIZE = 20

def extract_json(text):
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None

    json_str = match.group(0)
    json_str = json_str.replace("\n", " ")

    try:
        return json.loads(json_str)
    except:
        return None

def generate_batch(n, include_labels=True):
    prompt = f"""
Generate {n} realistic customer support tickets.

STRICT RULES:
- Return ONLY valid JSON array
- No markdown, no explanation
- Use double quotes only

Each item must have:
- ticket_id
- subject
- message
{"- category (billing, technical_issue, account_access, feature_request, general_query)" if include_labels else ""}
{"- priority (low, medium, high)" if include_labels else ""}

Example format:
[
  {{
    "ticket_id": 1,
    "subject": "...",
    "message": "...",
    "category": "billing",
    "priority": "high"
  }}
]
"""

    response = ai.generate_text(prompt)
    data = extract_json(response)

    if data is None:
        print("⚠️ Retry batch...")
        time.sleep(1)
        return generate_batch(n, include_labels)

    return data

def generate_dataset(total_size, include_labels=True):
    all_data = []

    for i in range(total_size // BATCH_SIZE):
        batch = generate_batch(BATCH_SIZE, include_labels)

        for j, item in enumerate(batch):
            item["ticket_id"] = len(all_data) + j

        all_data.extend(batch)
        print(f"✅ Generated {len(all_data)} samples")

    return all_data

train_data = generate_dataset(TRAIN_SIZE, include_labels=True)
train_df = pd.DataFrame(train_data)
train_df.to_csv("tickets_train.csv", index=False)

eval_data = generate_dataset(EVAL_SIZE, include_labels=False)
eval_df = pd.DataFrame(eval_data)

eval_df = eval_df.drop(columns=["category", "priority"], errors="ignore")
eval_df.to_csv("tickets_eval.csv", index=False)

kb_prompt = """
Create a small SaaS knowledge base.

STRICT RULES:
- Return ONLY valid JSON
- No markdown

Format:
{
  "billing.txt": "...",
  "technical.txt": "...",
  "account.txt": "...",
  "features.txt": "...",
  "general.txt": "..."
}

Each document should be 5–6 sentences explaining the topic.
"""

kb_response = ai.generate_text(kb_prompt)

match = re.search(r"\{.*\}", kb_response, re.DOTALL)
kb_data = json.loads(match.group(0))

os.makedirs("kb", exist_ok=True)

for filename, content in kb_data.items():
    with open(f"kb/{filename}", "w") as f:
        f.write(content)

print("\n🎉 DONE!")
print("Generated:")
print("- tickets_train.csv")
print("- tickets_eval.csv")
print("- kb/ folder")

✅ Generated 20 samples
✅ Generated 40 samples
✅ Generated 60 samples
✅ Generated 80 samples
✅ Generated 100 samples
✅ Generated 120 samples
✅ Generated 140 samples
✅ Generated 160 samples
✅ Generated 180 samples
✅ Generated 200 samples
✅ Generated 20 samples
✅ Generated 40 samples

🎉 DONE!
Generated:
- tickets_train.csv
- tickets_eval.csv
- kb/ folder


In [4]:
import pandas as pd
df = pd.read_csv("/content/tickets_train.csv")
df

,ticket_id,subject,message,category,priority
0,0,Payment Failed - Service Suspended,My recent payment for invoice #INV-2024-03-01 ...,billing,high
1,1,Website Loading Very Slowly,"For the past two days, your website has been e...",technical_issue,medium
2,2,Cannot Log In - Password Reset Not Working,I forgot my password and tried to reset it mul...,account_access,high
3,3,Feature Request: Dark Mode Option,I spend many hours on your platform daily. A d...,feature_request,low
4,4,How to Upgrade My Subscription Plan?,I'm currently on the 'Basic' plan and would li...,general_query,medium
...,...,...,...,...,...
195,195,Search Function Not Yielding Correct Results,The search function within the platform isn't ...,technical_issue,medium
196,196,Still Charged After Subscription Cancellation,"I cancelled my subscription on November 15th, ...",billing,high
197,197,Request to Merge Two Accounts,I accidentally created a second account (email...,account_access,medium
198,198,Feature Request: Customizable Dashboard Widgets,"The current dashboard is useful, but it would ...",feature_request,medium


In [3]:
import pandas as pd

train_df = pd.read_csv("tickets_train.csv")
eval_df = pd.read_csv("tickets_eval.csv")

train_df.head()

,ticket_id,subject,message,category,priority
0,0,Incorrect charge on my last invoice,I was charged $49.99 but my subscription is fo...,billing,high
1,1,Application crashing repeatedly on iOS,The mobile app crashes every time I try to ope...,technical_issue,high
2,2,Cannot log in to my account,I'm unable to log in to my account. I've tried...,account_access,medium
3,3,Request for Dark Mode feature,Could you please consider adding a dark mode o...,feature_request,low
4,4,How to export data to CSV?,I need to export all my transaction data for t...,general_query,medium


In [27]:
import pandas as pd

pred_df = pd.DataFrame(results)
pred_df.to_csv("predictions.csv", index=False)

pred_df.head()

,ticket_id,predicted_category,predicted_priority,top_3_retrieved_sources,draft_response,confidence_score,abstain_flag
0,0,billing,high,"billing.txt, account.txt, general.txt","For any billing disputes, please contact our s...",0.518375,False
1,1,account_access,high,"account.txt, billing.txt, general.txt","Should you face any login difficulties, please...",0.495150,False
2,2,technical_issue,medium,"technical.txt, billing.txt, general.txt",insufficient information,0.000000,True
3,3,general_query,low,"technical.txt, features.txt, general.txt",insufficient information,0.000000,True
4,4,technical_issue,medium,"general.txt, features.txt, billing.txt",insufficient information,0.000000,True
